In [0]:
dbutils.library.restartPython()

In [0]:
# =========================================
# 1. CONFIG PATH
# =========================================
dbutils.widgets.text("path", "")

config_path= dbutils.widgets.get("path")

In [0]:
%pip install pymongo
from common_utils.ingester import read_cosmosdb_json, write_raw
from common_utils.logging import get_logger
from datetime import datetime , date
import json
# =========================================
# 0.Logger
# =========================================
logger = get_logger('func_cosmosdb.ipynb')
now = datetime.now()
logger.info('session started at %s',now)






# =========================================
# 2. Read Config
# =========================================
with open(config_path, "r") as f:
  config= json.load(f)

# =========================================
# 3. CONFIG Sections
# =========================================
config_source = config["source"]
config_target = config["target"]
config_write_options = config["write_options"]



# =========================================
# 4. Run Date
# =========================================

run_date = date.today().isoformat()

# =========================================
# 5. Connection to MongoClient
# =========================================
connection_strin = dbutils.secrets.get(scope="retail-platform-dev",key=config_source["connection_string"])

# =========================================
# 6. Connection to MongoClient
# =========================================
df = read_cosmosdb_json(spark,connection_strin, config_source["database_name"] , config_source["collection_name"])
logger.info('rows in DataFrame %s', df.count())
df.display()
logger.info('printing sample data')

target_path =  f"{config_target["base_path"]}/{config_target["folder"]}/load_date={run_date}"
write_raw(df,target_path,config_target["mode"],config_target["file_format"], config_write_options["options"])
logger.info("Data frame is save to %s on date %s",target_path,  run_date)